# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, examine, and analyze the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_jsonld = dataset.metadata.to_json()
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We enumerate available `RecordSet`s. Each `RecordSet` corresponds to a data table or file as described by the schema. Their `@id` uniquely identifies them.

> Note: Replace `<record_set_id>` below with the actual `@id` you want to inspect.

In [ ]:
# List all record sets in the metadata by their @id
record_sets = [rs['@id'] for rs in getattr(dataset.metadata, 'recordSet', [])]
print('RecordSet @ids found in this dataset:')
for i, rs_id in enumerate(record_sets):
    print(f"{i+1}. {rs_id}")

if not record_sets:
    print('No explicit record sets found in the metadata.\n')
    # Try to infer available record sets from distributions if any
    distributions = getattr(dataset.metadata, 'distribution', [])
    print('Distributions (could act as record sets):')
    for dist in distributions:
        print(json.dumps(dist, indent=2))

# Print available fields for each record set (if present)
for rs in getattr(dataset.metadata, 'recordSet', []):
    print(f"\nRecordSet: {rs['@id']}")
    fields = rs.get('field', [])
    print('Fields:')
    for f in fields:
        if isinstance(f, dict):
            print(f"  - {f['@id']} ({f.get('name', '--no name--')})")
        else:
            print(f"  - {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

If no explicit record sets are present, you may use the dataset's first distribution as shown below.

In [ ]:
# --- If there are explicit recordSets (usually best practice, but not always present in real-world Croissant schemas)
if record_sets:
    # Choose the first record set for demonstration
    record_set_id = record_sets[0]
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records from RecordSet '{record_set_id}'. Columns:")
    print(df.columns.tolist())
    display(df.head())
else:
    # If no recordSet is present, try the first distribution as a record set
    if getattr(dataset.metadata, 'distribution', []):
        # There may be only one data table; Croissant schema allows direct records from the dataset
        records = list(dataset.records())
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records. Columns:")
        print(df.columns.tolist())
        display(df.head())
    else:
        print('No data record sets or distributions available for extraction.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This includes removing outliers, transforming data distributions, or grouping by key attributes for further analysis.

We'll pick some likely quantitative fields such as 'MW [kDa]' (molecular weight) or a peptide count column, by their schema `@id` or DataFrame column name.

In [ ]:
# Choose a numeric field
candidate_numeric_fields = [c for c in df.columns if (('count' in c.lower()) or ('mw' in c.lower()) or ('abundance' in c.lower()) or (df[c].dtype.kind in 'fi'))]
# Print available numeric-like candidates
print('Numeric fields:')
for c in candidate_numeric_fields:
    print(f"- {c}")

# For demonstration, pick the first candidate (edit as needed for precise field)
numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else df.columns[0]

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field]) else df.head(10)

print(f"\nFiltered records with '{numeric_field}' > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to group by a categorical field
candidate_group_fields = [c for c in df.columns if (('group' in c.lower()) or ('sample' in c.lower()) or ('type' in c.lower()))]
if candidate_group_fields:
    group_field = candidate_group_fields[0]
    # Group and average
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped data by '{group_field}':")
    display(grouped_df.head())
else:
    print('No obvious grouping field found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Plot the distribution of the selected numeric field
fig, ax = plt.subplots(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, ax=ax)
ax.set_title(f"Distribution of '{numeric_field}' field")
plt.xlabel(numeric_field)
plt.show()

# Optional: If two numeric fields, plot relation
if len(candidate_numeric_fields) >= 2:
    y_field = candidate_numeric_fields[1]
    fig, ax = plt.subplots(figsize=(6,6))
    sns.scatterplot(x=df[numeric_field], y=df[y_field], ax=ax)
    ax.set_title(f"Scatter: {numeric_field} vs. {y_field}")
    plt.xlabel(numeric_field)
    plt.ylabel(y_field)
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to use `mlcroissant` to load and explore the FAIR² dataset package:

- We loaded dataset metadata and inspected record sets and fields by their schema `@id`.
- We programmatically extracted the main data table into a DataFrame.
- We selected a numeric field (by name or type), filtered and normalized values, and briefly explored groupings and distributions.
- Simple visualizations helped summarize quantitive properties of the proteins measured.

**Next steps** might include:
- Deeper domain-specific analysis (e.g., peptide/protein occurrence by biological annotation),
- Outlier removal and correction,
- Complex visualizations (violin plots, volcano plots, cluster analysis, etc.),
- Integration with other proteomics or cell biology datasets.

> Analysis is powered by the Croissant schema and `mlcroissant`, referencing each entity by its `@id` for reproducible and transparent data science.